In [1]:
import sqlite3

In [2]:
conn = sqlite3.connect("school1.db")
conn.execute("PRAGMA foreign_keys = ON")  # Enforces foreign key constraints
cursor = conn.cursor()

In [3]:
cursor.execute("""     
    CREATE TABLE IF NOT EXISTS Coders (
        student_id INTEGER PRIMARY KEY,
        first_name TEXT NOT NULL,
        last_name TEXT NOT NULL,
        age INTEGER,
        grade TEXT,
        email TEXT UNIQUE
    )
""")
conn.commit()

In [4]:
cursor.execute("""
INSERT INTO Coders (student_id, first_name, last_name, age, grade, email)
VALUES (?, ?, ?, ?, ?, ?)
""", (100, "Cate", "Michael", 17, "10th", "cate.michael@gmail.com"))
conn.commit()

In [6]:
students_data = [
    (101, "Melody", "Bonareri", 17, "10th", "melody.bonareri@gmail.com"),
    (102, "John", "Kimani", 16, "9th", "johny.Kimani@gmail.com"),
    (103, "Mary", "Wanjiru", 18, "11th", "mary.wanjiru@gmail.com"),
    (104, "Kevin", "Otieno", 17, "10th", "kevin.otieno@gmail.com"),
    (105, "Susan", "Kamau", 15, "8th", "susan.kamau@gmail.com")
]

cursor.executemany("""
INSERT INTO Coders (student_id, first_name, last_name, age, grade, email)
VALUES (?, ?, ?, ?, ?, ?)
""", students_data)

conn.commit()

In [7]:
##Creating a second table for courses

cursor.execute("""
CREATE TABLE IF NOT EXISTS courses (
    course_id INTEGER PRIMARY KEY AUTOINCREMENT,
    course_name TEXT NOT NULL UNIQUE
)
""")
conn.commit()

In [9]:
courses = [
    ("Data Science",),
    ("Machine Learning",),
    ("Databases",),
    ("Mathematics",)
]

cursor.executemany("INSERT OR IGNORE INTO courses (course_name) VALUES (?)", courses)
conn.commit()

# show all courses
cursor.execute("SELECT * FROM courses ORDER BY course_id")
print(cursor.fetchall())

[(1, 'Data Science'), (2, 'Machine Learning'), (3, 'Databases'), (4, 'Mathematics')]


In [11]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS enrollment (
    student_id INTEGER NOT NULL,
    course_id INTEGER NOT NULL,
    enrolled_on TEXT DEFAULT (datetime('now')),
    PRIMARY KEY (student_id, course_id),
    FOREIGN KEY (student_id) REFERENCES Coders(student_id) ON DELETE CASCADE,
    FOREIGN KEY (course_id) REFERENCES courses(course_id) ON DELETE CASCADE
)
""")

conn.commit()

In [12]:
enrollment = [
    (100, 1),  # Student 1 enrolled in Course 1
    (101, 2),  # Student 2 enrolled in Course 2
    (102, 3),  # Student 3 enrolled in Course 3
    (103, 4),  # Student 4 enrolled in Course 4
    (104, 1),  # Student 5 enrolled in Course 1
    (105, 2)   # Student 6 enrolled in Course 2
   
]

cursor.executemany("""
INSERT INTO enrollment (student_id, course_id)
VALUES (?, ?)
""", enrollment)

conn.commit()
print("Enrollments added successfully!")


Enrollments added successfully!


In [15]:
cursor.execute("""
SELECT 
    s.first_name,
    s.last_name,
    c.course_name
FROM enrollment e
JOIN Coders s ON e.student_id = s.student_id
JOIN courses c ON e.course_id = c.course_id
""")

for row in cursor.fetchall():
    print(row)


('Cate', 'Michael', 'Data Science')
('Melody', 'Bonareri', 'Machine Learning')
('John', 'Kimani', 'Databases')
('Mary', 'Wanjiru', 'Mathematics')
('Kevin', 'Otieno', 'Data Science')
('Susan', 'Kamau', 'Machine Learning')
